In [54]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
from pathlib import Path
from datetime import date, timedelta
import math
import random

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn
from tqdm.auto import tqdm

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
CHECKPOINTS_PATH = PROJECT_PATH / "checkpoints"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VAL_START = date(2020, 9, 9)

K = 12
RRF_K = 20
HISTORY_DAYS = 56
HISTORY_TOP_N = 50
TOP_N = 100

SEED = 1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

Device: cuda


In [56]:
train = pl.scan_parquet(PROCESSED_PATH / "train.parquet")

validation_ground_truth = pl.read_parquet(
    PROCESSED_PATH / "validation_ground_truth.parquet"
)

article_mapping = pl.read_parquet(
    PROCESSED_PATH / "article_mapping.parquet"
)

customer_mapping = pl.read_parquet(
    PROCESSED_PATH / "customer_mapping.parquet"
)

NUM_ITEMS = article_mapping.height + 1

print("Validation users:", validation_ground_truth.height)
print("Items:", NUM_ITEMS - 1)

Validation users: 72019
Items: 105542


In [57]:
def average_precision_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    hits = 0
    score = 0.0

    for i, item in enumerate(predicted[:k]):
        if item in actual:
            hits += 1
            score += hits / (i + 1)

    return score / min(len(actual), k)


def recall_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    return len(actual.intersection(predicted[:k])) / min(len(actual), k)


def ndcg_at_k(actual, predicted, k=12):
    actual = set(actual)

    if not actual:
        return 0.0

    dcg = sum(
        1 / np.log2(i + 2)
        for i, item in enumerate(predicted[:k])
        if item in actual
    )

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(min(len(actual), k))
    )

    return dcg / idcg

In [58]:
actuals = validation_ground_truth["actual"].to_list()
catalog_size = NUM_ITEMS - 1

def evaluate_predictions(predictions):
    predictions_12 = [
        prediction[:K]
        for prediction in predictions
    ]

    return {
        "MAP@12": sum(
            average_precision_at_k(a, p, K)
            for a, p in zip(actuals, predictions_12)
        ) / len(actuals),

        "Recall@12": sum(
            recall_at_k(a, p, K)
            for a, p in zip(actuals, predictions_12)
        ) / len(actuals),

        "NDCG@12": sum(
            ndcg_at_k(a, p, K)
            for a, p in zip(actuals, predictions_12)
        ) / len(actuals),

        "Coverage": len({
            item
            for prediction in predictions_12
            for item in prediction
        }) / catalog_size
    }

In [59]:
validation_users = validation_ground_truth.select("customer_idx")

history = (
    train
    .filter(pl.col("t_dat") >= VAL_START - timedelta(days=HISTORY_DAYS))
    .join(validation_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"], descending=[False, True])
    .group_by("customer_idx", maintain_order=True)
    .agg(
        pl.col("article_idx")
        .unique(maintain_order=True)
        .head(HISTORY_TOP_N)
        .alias("history")
    )
    .collect()
)

history = (
    validation_users
    .join(history, on="customer_idx", how="left")
)

print("Users with history:", history["history"].is_not_null().sum())

Users with history: 44788


In [60]:
assert history["customer_idx"].to_list() == validation_ground_truth["customer_idx"].to_list()

history_lists = [
    items if items is not None else []
    for items in history["history"].to_list()
]

print("Tests passed")

Tests passed


In [61]:
DECAY_HALF_LIFE = 3

decay_top100 = (
    train
    .with_columns(
        (
            -np.log(2)
            * (
                (pl.lit(VAL_START) - pl.col("t_dat"))
                .dt.total_days()
            )
            / DECAY_HALF_LIFE
        )
        .exp()
        .alias("weight")
    )
    .group_by("article_idx")
    .agg(pl.col("weight").sum().alias("score"))
    .sort("score", descending=True)
    .head(TOP_N)
    .collect()["article_idx"]
    .to_list()
)

print(decay_top100[:12])

[103794, 67523, 67544, 53893, 104046, 103797, 3092, 94675, 101368, 101719, 103187, 71111]


In [62]:
sasrec_checkpoints = [
    path
    for path in CHECKPOINTS_PATH.glob("*.pt")
    if "sasrec" in path.name.lower()
    and "metadata" not in path.name.lower()
]

for path in sasrec_checkpoints:
    print(path.name)

assert sasrec_checkpoints, "SASRec checkpoint not found"

sasrec_best.pt


In [63]:
SASREC_PATH = sasrec_checkpoints[0]

checkpoint = torch.load(
    SASREC_PATH,
    map_location=DEVICE,
    weights_only=False
)

print("Checkpoint:", SASREC_PATH.name)
print("Epoch:", checkpoint.get("epoch"))
print("MAP@12:", checkpoint.get("metrics", {}).get("MAP@12"))

Checkpoint: sasrec_best.pt
Epoch: 8
MAP@12: 0.014934833159714998


In [64]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        max_len,
        hidden_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.item_embedding = nn.Embedding(
            num_items,
            hidden_dim,
            padding_idx=0
        )

        self.position_embedding = nn.Embedding(
            max_len,
            hidden_dim
        )

        self.dropout = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, input_items):
        seq_len = input_items.size(1)

        positions = torch.arange(
            seq_len,
            device=input_items.device
        ).unsqueeze(0)

        x = (
            self.item_embedding(input_items)
            * math.sqrt(self.hidden_dim)
        )

        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        padding_mask = input_items == 0

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_items.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        x = self.norm(x)

        return x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

In [65]:
MAX_LEN = checkpoint["max_len"]
HIDDEN_DIM = checkpoint["hidden_dim"]
NUM_HEADS = checkpoint["num_heads"]
NUM_LAYERS = checkpoint["num_layers"]
DROPOUT = checkpoint["dropout"]

sasrec = SASRec(
    num_items=NUM_ITEMS,
    max_len=MAX_LEN,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

sasrec.load_state_dict(
    checkpoint["model_state_dict"]
)

sasrec.eval()

print("Loaded SASRec")

Loaded SASRec


In [66]:
sasrec_history = (
    train
    .join(validation_users.lazy(), on="customer_idx", how="semi")
    .sort(["customer_idx", "t_dat"])
    .group_by("customer_idx", maintain_order=True)
    .agg(
        pl.col("article_idx")
        .tail(MAX_LEN)
        .alias("sequence")
    )
    .collect()
)

sasrec_history = validation_users.join(
    sasrec_history,
    on="customer_idx",
    how="left"
)

sequences = sasrec_history["sequence"].to_list()

print(
    "Users with sequence:",
    sum(sequence is not None for sequence in sequences)
)

Users with sequence: 66624


In [67]:
sequence_array = np.zeros(
    (len(sequences), MAX_LEN),
    dtype=np.int64
)

has_sequence = np.zeros(
    len(sequences),
    dtype=bool
)

for i, sequence in enumerate(sequences):
    if not sequence:
        continue

    sequence = sequence[-MAX_LEN:]

    sequence_array[
        i,
        -len(sequence):
    ] = sequence

    has_sequence[i] = True

print("Shape:", sequence_array.shape)
print("With sequence:", has_sequence.sum())

Shape: (72019, 64)
With sequence: 66624


In [68]:
SASREC_TOP100_PATH = (
    EMBEDDINGS_PATH
    / "sasrec_validation_top100.npz"
)

if SASREC_TOP100_PATH.exists():
    saved_sasrec = np.load(
        SASREC_TOP100_PATH
    )

    sasrec_top100 = saved_sasrec[
        "recommendations"
    ]

    print("Loaded:", sasrec_top100.shape)

In [69]:
if not SASREC_TOP100_PATH.exists():
    sasrec_top100 = np.tile(
        np.asarray(decay_top100, dtype=np.int32),
        (len(sequences), 1)
    )

    valid_indices = np.flatnonzero(has_sequence)

    item_embeddings = (
        sasrec.item_embedding.weight
        .detach()
        .to(torch.float16)
    )

    SASREC_BATCH_SIZE = 256

    for start in tqdm(
        range(0, len(valid_indices), SASREC_BATCH_SIZE),
        desc="SASRec retrieval"
    ):
        indices = valid_indices[start:start + SASREC_BATCH_SIZE]

        batch = torch.from_numpy(
            sequence_array[indices]
        ).to(DEVICE)

        with torch.inference_mode():
            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16,
                enabled=DEVICE.type == "cuda"
            ):
                hidden = sasrec(batch)
                user_embeddings = hidden[:, -1]
                scores = user_embeddings @ item_embeddings.T

            scores[:, 0] = -torch.inf

            top_items = torch.topk(
                scores,
                TOP_N,
                dim=1
            ).indices

        sasrec_top100[indices] = (
            top_items.cpu().numpy().astype(np.int32)
        )

    np.savez_compressed(
        SASREC_TOP100_PATH,
        customer_idx=validation_users["customer_idx"].to_numpy(),
        recommendations=sasrec_top100
    )

    print("Saved:", SASREC_TOP100_PATH)
    print("Shape:", sasrec_top100.shape)

SASRec retrieval:   0%|          | 0/261 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/embeddings/sasrec_validation_top100.npz
Shape: (72019, 100)


In [70]:
visual_file = np.load(
    EMBEDDINGS_PATH / "visual_validation_top100_weighted.npz"
)

visual_customer_idx = visual_file["customer_idx"]
visual_top100 = visual_file["recommendations"]

print("Visual:", visual_top100.shape)
print("Half-life:", visual_file["half_life"][0])

Visual: (72019, 100)
Half-life: 7


In [71]:
validation_customer_idx = validation_ground_truth["customer_idx"].to_numpy()

assert np.array_equal(
    validation_customer_idx,
    visual_customer_idx
)

assert sasrec_top100.shape == (
    len(validation_ground_truth),
    TOP_N
)

assert visual_top100.shape == (
    len(validation_ground_truth),
    TOP_N
)

print("All recommendation sources aligned")

All recommendation sources aligned


In [72]:
def fuse_rankings(
    history,
    sasrec,
    decay,
    visual=None,
    history_weight=1.0,
    sasrec_weight=0.15,
    decay_weight=0.15,
    visual_weight=0.0,
    top_n=100
):
    scores = {}

    for rank, item in enumerate(history, start=1):
        scores[item] = scores.get(item, 0.0) + (
            history_weight / (RRF_K + rank)
        )

    for rank, item in enumerate(sasrec, start=1):
        scores[item] = scores.get(item, 0.0) + (
            sasrec_weight / (RRF_K + rank)
        )

    for rank, item in enumerate(decay, start=1):
        scores[item] = scores.get(item, 0.0) + (
            decay_weight / (RRF_K + rank)
        )

    if visual is not None and visual_weight > 0:
        for rank, item in enumerate(visual, start=1):
            scores[item] = scores.get(item, 0.0) + (
                visual_weight / (RRF_K + rank)
            )

    return [
        item
        for item, _ in sorted(
            scores.items(),
            key=lambda pair: pair[1],
            reverse=True
        )[:top_n]
    ]

In [73]:
base_hybrid = [
    fuse_rankings(
        history=history_lists[i],
        sasrec=sasrec_top100[i],
        decay=decay_top100,
        history_weight=1.0,
        sasrec_weight=0.15,
        decay_weight=0.15
    )
    for i in tqdm(
        range(len(actuals)),
        desc="Base hybrid"
    )
]

Base hybrid:   0%|          | 0/72019 [00:00<?, ?it/s]

In [74]:
base_hybrid_metrics = evaluate_predictions(
    base_hybrid
)

base_hybrid_metrics

{'MAP@12': 0.027050122236696717,
 'Recall@12': 0.05545447993551966,
 'NDCG@12': np.float64(0.03957046415492013),
 'Coverage': 0.25315040457827215}

In [75]:
zero_visual_hybrid = [
    fuse_rankings(
        history=history_lists[i],
        sasrec=sasrec_top100[i],
        decay=decay_top100,
        visual=visual_top100[i],
        history_weight=1.0,
        sasrec_weight=0.15,
        decay_weight=0.15,
        visual_weight=0.0,
        top_n=K
    )
    for i in range(len(actuals))
]

zero_visual_map = sum(
    average_precision_at_k(a, p, K)
    for a, p in zip(actuals, zero_visual_hybrid)
) / len(actuals)

print("Base MAP@12:", base_hybrid_metrics["MAP@12"])
print("Visual weight 0 MAP@12:", zero_visual_map)

assert abs(
    zero_visual_map - base_hybrid_metrics["MAP@12"]
) < 1e-12

print("Test passed")

Base MAP@12: 0.027050122236696717
Visual weight 0 MAP@12: 0.027050122236696717
Test passed


In [76]:
visual_weights = [
    0.025,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30
]

visual_weight_results = []

for visual_weight in tqdm(
    visual_weights,
    desc="Visual weight search"
):
    predictions = [
        fuse_rankings(
            history=history_lists[i],
            sasrec=sasrec_top100[i],
            decay=decay_top100,
            visual=visual_top100[i],
            history_weight=1.0,
            sasrec_weight=0.15,
            decay_weight=0.15,
            visual_weight=visual_weight,
            top_n=K
        )
        for i in range(len(actuals))
    ]

    map12 = sum(
        average_precision_at_k(a, p, K)
        for a, p in zip(actuals, predictions)
    ) / len(actuals)

    visual_weight_results.append({
        "visual_weight": visual_weight,
        "MAP@12": map12
    })

visual_weight_results = (
    pl.DataFrame(visual_weight_results)
    .sort("MAP@12", descending=True)
)

visual_weight_results

Visual weight search:   0%|          | 0/8 [00:00<?, ?it/s]

visual_weight,MAP@12
f64,f64
0.05,0.027201
0.025,0.027171
0.075,0.027151
0.1,0.027086
0.15,0.026969
0.2,0.026495
0.25,0.025945
0.3,0.025487


In [77]:
coarse_best_weight = visual_weight_results["visual_weight"][0]

fine_visual_weights = np.arange(
    max(0.01, coarse_best_weight - 0.04),
    coarse_best_weight + 0.041,
    0.01
)

fine_results = []

for visual_weight in tqdm(
    fine_visual_weights,
    desc="Fine visual weight search"
):
    predictions = [
        fuse_rankings(
            history=history_lists[i],
            sasrec=sasrec_top100[i],
            decay=decay_top100,
            visual=visual_top100[i],
            history_weight=1.0,
            sasrec_weight=0.15,
            decay_weight=0.15,
            visual_weight=float(visual_weight),
            top_n=K
        )
        for i in range(len(actuals))
    ]

    map12 = sum(
        average_precision_at_k(a, p, K)
        for a, p in zip(actuals, predictions)
    ) / len(actuals)

    fine_results.append({
        "visual_weight": float(visual_weight),
        "MAP@12": map12
    })

fine_results = (
    pl.DataFrame(fine_results)
    .sort("MAP@12", descending=True)
)

fine_results

Fine visual weight search:   0%|          | 0/9 [00:00<?, ?it/s]

visual_weight,MAP@12
f64,f64
0.04,0.027226
0.05,0.027201
0.06,0.027194
0.03,0.027181
0.08,0.027154
0.02,0.027146
0.07,0.027142
0.09,0.027127
0.01,0.027119


In [78]:
VISUAL_WEIGHT = fine_results["visual_weight"][0]

print("Best visual weight:", VISUAL_WEIGHT)
print("Best MAP@12:", fine_results["MAP@12"][0])
print("Base MAP@12:", base_hybrid_metrics["MAP@12"])

Best visual weight: 0.04000000000000001
Best MAP@12: 0.027225635411029264
Base MAP@12: 0.027050122236696717


In [79]:
final_hybrid = [
    fuse_rankings(
        history=history_lists[i],
        sasrec=sasrec_top100[i],
        decay=decay_top100,
        visual=visual_top100[i],
        history_weight=1.0,
        sasrec_weight=0.15,
        decay_weight=0.15,
        visual_weight=VISUAL_WEIGHT,
        top_n=TOP_N
    )
    for i in tqdm(
        range(len(actuals)),
        desc="Final hybrid"
    )
]

final_hybrid_metrics = evaluate_predictions(
    final_hybrid
)

final_hybrid_metrics

Final hybrid:   0%|          | 0/72019 [00:00<?, ?it/s]

{'MAP@12': 0.027225635411029264,
 'Recall@12': 0.0560012647576346,
 'NDCG@12': np.float64(0.03985322083557821),
 'Coverage': 0.23989501809706087}

In [80]:
comparison = pl.DataFrame([
    {
        "model": "History + SASRec + DecayPop",
        **base_hybrid_metrics
    },
    {
        "model": "History + SASRec + DecayPop + CLIP",
        **final_hybrid_metrics
    }
])

comparison

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""History + SASRec + DecayPop""",0.02705,0.055454,0.03957,0.25315
"""History + SASRec + DecayPop + …",0.027226,0.056001,0.039853,0.239895


In [81]:
train_item_ids = set(
    train
    .select("article_idx")
    .unique()
    .collect()["article_idx"]
    .to_list()
)

cold_user_indices = []
cold_targets = []

for i, actual in enumerate(actuals):
    cold_actual = [
        item
        for item in actual
        if item not in train_item_ids
    ]

    if cold_actual:
        cold_user_indices.append(i)
        cold_targets.append(cold_actual)

print("Cold-start users:", len(cold_user_indices))

Cold-start users: 8418


In [82]:
def evaluate_cold(predictions):
    cold_predictions_12 = [
        predictions[i][:K]
        for i in cold_user_indices
    ]

    cold_predictions_100 = [
        predictions[i][:100]
        for i in cold_user_indices
    ]

    return {
        "Cold MAP@12": sum(
            average_precision_at_k(a, p, K)
            for a, p in zip(
                cold_targets,
                cold_predictions_12
            )
        ) / len(cold_targets),

        "Cold Recall@12": sum(
            recall_at_k(a, p, K)
            for a, p in zip(
                cold_targets,
                cold_predictions_12
            )
        ) / len(cold_targets),

        "Cold Recall@100": sum(
            recall_at_k(a, p, 100)
            for a, p in zip(
                cold_targets,
                cold_predictions_100
            )
        ) / len(cold_targets)
    }

In [83]:
base_cold_metrics = evaluate_cold(base_hybrid)
final_cold_metrics = evaluate_cold(final_hybrid)

pl.DataFrame([
    {
        "model": "History + SASRec + DecayPop",
        **base_cold_metrics
    },
    {
        "model": "History + SASRec + DecayPop + CLIP",
        **final_cold_metrics
    }
])

model,Cold MAP@12,Cold Recall@12,Cold Recall@100
str,f64,f64,f64
"""History + SASRec + DecayPop""",0.0,0.0,0.0
"""History + SASRec + DecayPop + …",0.0,0.0,0.0


In [84]:
sasrec_weights = [0.10, 0.15, 0.20]
decay_weights = [0.10, 0.15, 0.20]
visual_weights = [0.02, 0.03, 0.04, 0.05, 0.06]

joint_results = []

for sasrec_weight in sasrec_weights:
    for decay_weight in decay_weights:
        for visual_weight in visual_weights:
            predictions = [
                fuse_rankings(
                    history=history_lists[i],
                    sasrec=sasrec_top100[i],
                    decay=decay_top100,
                    visual=visual_top100[i],
                    history_weight=1.0,
                    sasrec_weight=sasrec_weight,
                    decay_weight=decay_weight,
                    visual_weight=visual_weight,
                    top_n=K
                )
                for i in range(len(actuals))
            ]

            map12 = sum(
                average_precision_at_k(a, p, K)
                for a, p in zip(
                    actuals,
                    predictions
                )
            ) / len(actuals)

            joint_results.append({
                "sasrec_weight": sasrec_weight,
                "decay_weight": decay_weight,
                "visual_weight": visual_weight,
                "MAP@12": map12
            })

joint_results = (
    pl.DataFrame(joint_results)
    .sort("MAP@12", descending=True)
)

joint_results

sasrec_weight,decay_weight,visual_weight,MAP@12
f64,f64,f64,f64
0.15,0.1,0.06,0.027333
0.15,0.1,0.05,0.027313
0.2,0.15,0.06,0.0273
0.2,0.15,0.05,0.027242
0.15,0.15,0.04,0.027226
…,…,…,…
0.1,0.2,0.06,0.026409
0.1,0.2,0.05,0.026407
0.1,0.2,0.04,0.026384


In [85]:
from itertools import product

sasrec_weights = [0.12, 0.15, 0.18]
decay_weights = [0.04, 0.06, 0.08, 0.10]
visual_weights = [0.05, 0.06, 0.07, 0.08, 0.10, 0.12]

weight_grid = list(
    product(
        sasrec_weights,
        decay_weights,
        visual_weights
    )
)

fine_joint_results = []

for sasrec_weight, decay_weight, visual_weight in tqdm(
    weight_grid,
    desc="Fine joint search"
):
    predictions = [
        fuse_rankings(
            history=history_lists[i],
            sasrec=sasrec_top100[i],
            decay=decay_top100,
            visual=visual_top100[i],
            history_weight=1.0,
            sasrec_weight=sasrec_weight,
            decay_weight=decay_weight,
            visual_weight=visual_weight,
            top_n=K
        )
        for i in range(len(actuals))
    ]

    map12 = sum(
        average_precision_at_k(a, p, K)
        for a, p in zip(actuals, predictions)
    ) / len(actuals)

    fine_joint_results.append({
        "sasrec_weight": sasrec_weight,
        "decay_weight": decay_weight,
        "visual_weight": visual_weight,
        "MAP@12": map12
    })

fine_joint_results = (
    pl.DataFrame(fine_joint_results)
    .sort("MAP@12", descending=True)
)

fine_joint_results

Fine joint search:   0%|          | 0/72 [00:00<?, ?it/s]

sasrec_weight,decay_weight,visual_weight,MAP@12
f64,f64,f64,f64
0.12,0.08,0.07,0.027367
0.12,0.08,0.08,0.027356
0.18,0.1,0.12,0.027354
0.12,0.08,0.05,0.027352
0.15,0.1,0.08,0.027346
…,…,…,…
0.18,0.04,0.07,0.026802
0.18,0.06,0.05,0.026745
0.18,0.04,0.06,0.026744


In [86]:
best_weights = fine_joint_results.row(
    0,
    named=True
)

SASREC_WEIGHT = best_weights["sasrec_weight"]
DECAY_WEIGHT = best_weights["decay_weight"]
VISUAL_WEIGHT = best_weights["visual_weight"]

print("SASRec weight:", SASREC_WEIGHT)
print("Decay weight:", DECAY_WEIGHT)
print("Visual weight:", VISUAL_WEIGHT)
print("MAP@12:", best_weights["MAP@12"])

SASRec weight: 0.12
Decay weight: 0.08
Visual weight: 0.07
MAP@12: 0.02736748464454252


In [87]:
final_hybrid = [
    fuse_rankings(
        history=history_lists[i],
        sasrec=sasrec_top100[i],
        decay=decay_top100,
        visual=visual_top100[i],
        history_weight=1.0,
        sasrec_weight=SASREC_WEIGHT,
        decay_weight=DECAY_WEIGHT,
        visual_weight=VISUAL_WEIGHT,
        top_n=TOP_N
    )
    for i in tqdm(
        range(len(actuals)),
        desc="Final validation hybrid"
    )
]

final_hybrid_metrics = evaluate_predictions(
    final_hybrid
)

final_hybrid_metrics

Final validation hybrid:   0%|          | 0/72019 [00:00<?, ?it/s]

{'MAP@12': 0.02736748464454252,
 'Recall@12': 0.056280505042290124,
 'NDCG@12': np.float64(0.03997301839113418),
 'Coverage': 0.24367550359098747}

In [88]:
final_cold_metrics = evaluate_cold(
    final_hybrid
)

final_cold_metrics

{'Cold MAP@12': 0.0,
 'Cold Recall@12': 0.0,
 'Cold Recall@100': 0.003252101505843487}

In [89]:
FINAL_WEIGHTS_PATH = PROCESSED_PATH / "final_hybrid_weights.npz"

np.savez(
    FINAL_WEIGHTS_PATH,
    history_weight=np.float32(1.0),
    sasrec_weight=np.float32(SASREC_WEIGHT),
    decay_weight=np.float32(DECAY_WEIGHT),
    visual_weight=np.float32(VISUAL_WEIGHT),
    validation_map12=np.float32(final_hybrid_metrics["MAP@12"]),
    validation_recall12=np.float32(final_hybrid_metrics["Recall@12"]),
    validation_ndcg12=np.float32(final_hybrid_metrics["NDCG@12"]),
    validation_coverage=np.float32(final_hybrid_metrics["Coverage"])
)

print("Saved:", FINAL_WEIGHTS_PATH)

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/data/processed/final_hybrid_weights.npz


In [90]:
final_comparison = pl.DataFrame([
    {
        "model": "History + SASRec + DecayPop",
        **base_hybrid_metrics
    },
    {
        "model": "History + SASRec + DecayPop + CLIP",
        **final_hybrid_metrics
    }
])

final_comparison

model,MAP@12,Recall@12,NDCG@12,Coverage
str,f64,f64,f64,f64
"""History + SASRec + DecayPop""",0.02705,0.055454,0.03957,0.25315
"""History + SASRec + DecayPop + …",0.027367,0.056281,0.039973,0.243676
